# Notebook 07b: Transfer Learning with PyTorch

---

## Overview

This notebook implements **transfer learning using PyTorch** for multi-label chest X-ray disease classification. This complements notebook 07 (TensorFlow) and demonstrates framework-appropriate tool selection.

**Objectives:**
1. Implement PyTorch Dataset and DataLoader for medical imaging
2. Leverage `timm` library for access to 500+ pre-trained models
3. Train three transfer learning models: DenseNet121, ConvNeXt-Tiny, EfficientNetV2-S
4. Implement two-stage training: feature extraction → fine-tuning
5. Compare PyTorch vs TensorFlow implementations

**Why PyTorch for Notebook 07b?**

While notebook 07 uses TensorFlow for transfer learning, PyTorch offers distinct advantages:

- **timm Library**: Access to 500+ cutting-edge pre-trained models vs ~20 in keras.applications
- **Modern Architectures**: ConvNeXt (2022), EfficientNetV2, RegNet unavailable in TensorFlow
- **Medical Imaging Ecosystem**: MONAI (Medical Open Network for AI) built on PyTorch
- **Research Alignment**: 70-80% of recent medical imaging papers use PyTorch
- **Explicit Control**: Manual training loops provide better understanding of process
- **Pedagogical Value**: Demonstrates framework versatility and informed tool selection

**Framework Comparison:**

| Aspect | TensorFlow (NB 07) | PyTorch (NB 07b) |
|--------|-------------------|------------------|
| **API Style** | High-level (model.fit) | Low-level (manual loops) |
| **Model Zoo** | ~20 models | 500+ models (timm) |
| **Medical AI** | Limited | MONAI ecosystem |
| **Learning Curve** | Easier (more automation) | Steeper (more explicit) |
| **Research Trends** | Declining | Growing |
| **Production** | TensorFlow Serving | TorchServe, ONNX |

**Assessment Alignment:**

This dual-framework approach demonstrates:
- **LO11**: Adaptation and use of diverse tools (PyTorch + TensorFlow)
- **Technical Depth**: Understanding of different deep learning paradigms
- **Critical Thinking**: Selecting appropriate framework based on task requirements
- **Modern Skills**: PyTorch expertise increasingly required in ML roles

**Outputs:**
- Fine-tuned PyTorch models (DenseNet121, ConvNeXt, EfficientNetV2)
- Performance comparison: PyTorch vs TensorFlow vs Baseline models
- Framework selection insights for future work

---

## 1. Setup and Configuration

In [ ]:
# Import standard libraries
import json
import sys
import warnings
from pathlib import Path
from datetime import datetime

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# PyTorch vision
from torchvision import transforms
from PIL import Image

# timm - PyTorch Image Models
import timm

# Metrics (sklearn for consistency with previous notebooks)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Progress bars
from tqdm.auto import tqdm

# MLflow tracking
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✓ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"timm version: {timm.__version__}")
print(f"MLflow version: {mlflow.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"\nUsing device: {device}")

In [ ]:
# Define paths
current_path = Path.cwd()

if current_path.name == 'jupyter_notebooks':
    PROJECT_ROOT = current_path.parent
elif (current_path / 'setup.py').exists() or (current_path / 'README.md').exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models' / 'saved_models'

print(f"Project root: {PROJECT_ROOT}")
print(f"Models directory: {MODELS_DIR}")

In [ ]:
# PyTorch Transfer Learning Configuration
CONFIG = {
    # Image parameters
    'img_height': 224,
    'img_width': 224,
    'channels': 3,
    
    # Training parameters - Stage 1 (Feature extraction)
    'batch_size': 32,
    'epochs_stage1': 10,  # Feature extraction
    'learning_rate_stage1': 0.001,
    
    # Training parameters - Stage 2 (Fine-tuning)
    'epochs_stage2': 20,  # Fine-tuning
    'learning_rate_stage2': 0.0001,  # 10x lower LR for fine-tuning
    
    # Model architecture
    'dropout_rate': 0.5,
    
    # Callbacks
    'early_stopping_patience': 10,
    'reduce_lr_patience': 5,
    'reduce_lr_factor': 0.5,
    
    # Data
    'num_classes': 14,
    'use_sample': True,
    'sample_size': 5000,
    
    'random_state': 42,
    
    # PyTorch specific
    'num_workers': 4,  # DataLoader workers
    'pin_memory': True if torch.cuda.is_available() else False,
}

print("PyTorch Transfer Learning Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Parameters (can be overridden by papermill)
# This cell is tagged as "parameters" for papermill
MODELS_TO_TRAIN = ['densenet121', 'convnext_tiny', 'efficientnetv2_s']  # Which models to train
RUN_NAME = None  # Custom run name for MLflow (auto-generated if None)
USE_MLFLOW = True  # Enable MLflow tracking

print(f"Models to train: {MODELS_TO_TRAIN}")
print(f"Run name: {RUN_NAME if RUN_NAME else 'Auto-generated'}")
print(f"MLflow tracking: {'Enabled' if USE_MLFLOW else 'Disabled'}")

## Training Control

In [ ]:
# Training control flag
RETRAIN_MODELS = False

print(f"Training Mode: {'RETRAIN ALL' if RETRAIN_MODELS else 'USE SAVED'}")

## MLflow Setup

In [ ]:
# Setup MLflow tracking
if USE_MLFLOW:
    # Set tracking URI to local SQLite database
    mlflow_db_path = PROJECT_ROOT / 'mlflow.db'
    mlflow.set_tracking_uri(f"sqlite:///{mlflow_db_path}")
    
    # Set experiment
    experiment_name = "07b-transfer-learning-pytorch"
    mlflow.set_experiment(experiment_name)
    
    # Generate run name if not provided
    if RUN_NAME is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        RUN_NAME = f"pytorch_transfer_{timestamp}"
    
    print(f"✓ MLflow tracking configured")
    print(f"  Tracking URI: sqlite:///{mlflow_db_path}")
    print(f"  Experiment: {experiment_name}")
    print(f"  Run name: {RUN_NAME}")
else:
    print("⚠️  MLflow tracking disabled")

## 2. Load Data

In [ ]:
# Load split files
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']
class_weights_dict = prep_config['class_weights']

print(f"✓ Loaded data splits and configuration")
print(f"  Disease classes: {len(disease_classes)}")
print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

In [ ]:
# Sample if configured
if CONFIG['use_sample']:
    sample_size = CONFIG['sample_size']
    train_df = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    val_df = val_df.sample(n=min(sample_size // 5, len(val_df)), random_state=42)
    test_df = test_df.sample(n=min(sample_size // 5, len(test_df)), random_state=42)
    
    print(f"⚠️ Using sample mode:")
    print(f"  Train: {len(train_df):,} images")
    print(f"  Val:   {len(val_df):,} images")
    print(f"  Test:  {len(test_df):,} images")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 3. PyTorch Dataset and DataLoader

### 📚 PyTorch Data Pipeline

Unlike TensorFlow's `ImageDataGenerator`, PyTorch uses:

1. **Dataset**: Custom class defining how to load individual samples
2. **DataLoader**: Batch loading with multi-processing, shuffling, etc.

**Advantages:**
- More explicit and pythonic
- Better multi-processing support
- Easier to debug (standard Python classes)
- More flexible transformations

---

In [ ]:
class ChestXrayDataset(Dataset):
    """
    PyTorch Dataset for NIH Chest X-Ray multi-label classification.
    
    Args:
        dataframe: pandas DataFrame with 'full_path' and disease columns
        disease_classes: List of disease column names
        transform: torchvision.transforms composition (default: None)
    """
    def __init__(self, dataframe, disease_classes, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.disease_classes = disease_classes
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.df.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        # Get labels (multi-label: 14 diseases)
        labels = self.df.iloc[idx][self.disease_classes].values.astype('float32')
        
        # Apply transformations
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(labels, dtype=torch.float32)

print("✓ ChestXrayDataset class defined")

In [ ]:
# Define transformations (equivalent to ImageDataGenerator)

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_height'], CONFIG['img_width'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # width/height shift
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),  # Converts to [0, 1] and changes to CHW format
])

# Validation/Test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((CONFIG['img_height'], CONFIG['img_width'])),
    transforms.ToTensor(),
])

print("✓ Transforms defined")

In [ ]:
# Create Dataset instances
train_dataset = ChestXrayDataset(train_df, disease_classes, transform=train_transform)
val_dataset = ChestXrayDataset(val_df, disease_classes, transform=val_transform)
test_dataset = ChestXrayDataset(test_df, disease_classes, transform=val_transform)

print(f"✓ Datasets created:")
print(f"  Train: {len(train_dataset):,} samples")
print(f"  Val:   {len(val_dataset):,} samples")
print(f"  Test:  {len(test_dataset):,} samples")

# Test dataset
sample_img, sample_labels = train_dataset[0]
print(f"\nSample check:")
print(f"  Image shape: {sample_img.shape} (C×H×W format)")
print(f"  Labels shape: {sample_labels.shape}")
print(f"  Image range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory'],
    drop_last=True  # Drop incomplete batches for consistent training
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory']
)

print(f"✓ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

## 4. Training Infrastructure

### 📚 PyTorch Training Loop

Unlike TensorFlow's `model.fit()`, PyTorch requires manual training loops. This provides:
- **Explicit control** over every step
- **Easier debugging** (standard Python control flow)
- **Flexibility** for custom training logic

**Basic Pattern:**
```python
for epoch in range(epochs):
    model.train()  # Set to training mode
    for batch in train_loader:
        optimizer.zero_grad()  # Reset gradients
        loss = criterion(model(batch), labels)
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights
```

---

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    Train model for one epoch.
    
    Returns:
        Average loss and accuracy for the epoch
    """
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item() * images.size(0)
        all_preds.append(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item()})
    
    # Calculate epoch metrics
    epoch_loss = running_loss / len(loader.dataset)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Accuracy (using 0.5 threshold)
    epoch_acc = accuracy_score(all_labels.ravel(), (all_preds > 0.5).ravel())
    
    return epoch_loss, epoch_acc

print("✓ train_one_epoch function defined")

In [ ]:
def validate(model, loader, criterion, device):
    """
    Validate model.
    
    Returns:
        Average loss, accuracy, and AUC for validation set
    """
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():  # No gradient computation during validation
        for images, labels in tqdm(loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass only
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Track metrics
            running_loss += loss.item() * images.size(0)
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    # Calculate metrics
    val_loss = running_loss / len(loader.dataset)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    val_acc = accuracy_score(all_labels.ravel(), (all_preds > 0.5).ravel())
    val_auc = roc_auc_score(all_labels, all_preds, average='macro')
    
    return val_loss, val_acc, val_auc

print("✓ validate function defined")

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                num_epochs, device, save_path, model_name="model", stage="full", patience=10, 
                use_mlflow=True):
    """
    Complete training loop with early stopping, checkpointing, and MLflow tracking.
    
    Args:
        model: PyTorch model
        train_loader, val_loader: DataLoaders
        criterion: Loss function
        optimizer: Optimizer
        scheduler: Learning rate scheduler
        num_epochs: Number of epochs
        device: torch.device
        save_path: Path to save best model
        model_name: Name for MLflow tracking (e.g., "densenet121")
        stage: Training stage ("stage1" or "stage2")
        patience: Early stopping patience
        use_mlflow: Enable MLflow logging
        
    Returns:
        Dictionary with training history
    """
    best_val_auc = 0.0
    epochs_no_improve = 0
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_auc': [],
        'lr': []
    }
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 60)
        
        # Training
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validation
        val_loss, val_acc, val_auc = validate(model, val_loader, criterion, device)
        
        # Learning rate step
        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        history['lr'].append(current_lr)
        
        # Log to MLflow
        if use_mlflow and USE_MLFLOW:
            mlflow.log_metrics({
                f'{model_name}_{stage}_train_loss': train_loss,
                f'{model_name}_{stage}_train_acc': train_acc,
                f'{model_name}_{stage}_val_loss': val_loss,
                f'{model_name}_{stage}_val_acc': val_acc,
                f'{model_name}_{stage}_val_auc': val_auc,
                f'{model_name}_{stage}_lr': current_lr,
            }, step=epoch)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
        print(f"LR: {current_lr:.6f}")
        
        # Save best model
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), save_path)
            print(f"✓ Saved best model (AUC: {best_val_auc:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        # Early stopping
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping after {epoch+1} epochs (no improvement for {patience} epochs)")
            break
    
    # Load best model
    model.load_state_dict(torch.load(save_path))
    print(f"\n✓ Training complete. Best Val AUC: {best_val_auc:.4f}")
    
    # Log best metrics to MLflow
    if use_mlflow and USE_MLFLOW:
        mlflow.log_metrics({
            f'{model_name}_{stage}_best_val_auc': best_val_auc,
            f'{model_name}_{stage}_epochs_trained': len(history['train_loss']),
        })
    
    return history

print("✓ train_model function defined")

## 5. Build Transfer Learning Models with timm

### 📚 About timm (PyTorch Image Models)

**timm** is Ross Wightman's library providing:
- **500+ pre-trained models** (vs ~20 in keras.applications)
- Latest architectures: ConvNeXt, MaxViT, EfficientNetV2, etc.
- Easy model creation: `timm.create_model()`
- Unified interface across all models

**Model Selection for This Project:**

1. **DenseNet121** (baseline, compare with TF notebook 07)
2. **ConvNeXt-Tiny** (modern ConvNet, 2022)
3. **EfficientNetV2-S** (improved EfficientNet, 2021)

---

In [ ]:
# List available models
print("Available timm models (sample):")
print("\nDenseNet variants:")
print([m for m in timm.list_models('densenet*')][:5])
print("\nConvNeXt variants:")
print([m for m in timm.list_models('convnext*')][:5])
print("\nEfficientNetV2 variants:")
print([m for m in timm.list_models('efficientnetv2*')][:5])

In [ ]:
def create_transfer_model(model_name, num_classes=14, pretrained=True):
    """
    Create transfer learning model using timm.
    
    Args:
        model_name: Name of timm model (e.g., 'densenet121')
        num_classes: Number of output classes
        pretrained: Use ImageNet pre-trained weights
        
    Returns:
        PyTorch model ready for training
    """
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=num_classes,
        drop_rate=CONFIG['dropout_rate']  # Dropout before classifier
    )
    
    return model

print("✓ create_transfer_model function defined")

## 6. Check for Existing Models

In [ ]:
# Check if models exist
densenet_path = MODELS_DIR / "densenet121_pytorch_best.pt"
convnext_path = MODELS_DIR / "convnext_tiny_pytorch_best.pt"
efficientnet_path = MODELS_DIR / "efficientnetv2_s_pytorch_best.pt"

all_models_exist = all([
    densenet_path.exists(),
    convnext_path.exists(),
    efficientnet_path.exists()
])

if not RETRAIN_MODELS and all_models_exist:
    print("="*60)
    print("LOADING EXISTING MODELS")
    print("="*60)
    print("\nAll 3 PyTorch transfer learning models found. Loading...")
    print(f"  - {densenet_path}")
    print(f"  - {convnext_path}")
    print(f"  - {efficientnet_path}")
    print("\nSkipping training (RETRAIN_MODELS=False)")
    print("\n💡 Set RETRAIN_MODELS=True to retrain models\n")
    
    SKIP_TRAINING = True
else:
    if RETRAIN_MODELS:
        print("⚠️  RETRAIN_MODELS=True, training all models from scratch...")
    else:
        print("⚠️  Some models missing, will train all models...")
        missing = []
        if not densenet_path.exists(): missing.append("DenseNet121")
        if not convnext_path.exists(): missing.append("ConvNeXt-Tiny")
        if not efficientnet_path.exists(): missing.append("EfficientNetV2-S")
        print(f"     Missing: {', '.join(missing)}")
    
    SKIP_TRAINING = False
    print("\n▶️  Proceeding with training (this will take 30-90 min on GPU)...\n")

## 7. Model 1: DenseNet121

Baseline model for comparison with TensorFlow notebook 07.

**Architecture**: 121 layers, ~8M parameters, dense connections

---

In [ ]:
if not SKIP_TRAINING and 'densenet121' in MODELS_TO_TRAIN:
    print("="*60)
    print("MODEL 1: DenseNet121 (PyTorch)")
    print("="*60)
    
    # Start MLflow run for DenseNet121
    if USE_MLFLOW:
        mlflow_run = mlflow.start_run(run_name=f"{RUN_NAME}_densenet121")
        mlflow.log_params({
            'model': 'densenet121',
            'framework': 'pytorch',
            'batch_size': CONFIG['batch_size'],
            'epochs_stage1': CONFIG['epochs_stage1'],
            'epochs_stage2': CONFIG['epochs_stage2'],
            'lr_stage1': CONFIG['learning_rate_stage1'],
            'lr_stage2': CONFIG['learning_rate_stage2'],
            'dropout_rate': CONFIG['dropout_rate'],
            'use_sample': CONFIG['use_sample'],
        })
    
    # Create model
    densenet_model = create_transfer_model('densenet121', num_classes=CONFIG['num_classes'])
    densenet_model = densenet_model.to(device)
    
    print(f"\nModel: DenseNet121")
    print(f"Parameters: {sum(p.numel() for p in densenet_model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in densenet_model.parameters() if p.requires_grad):,}")
    
    # ============================================================
    # STAGE 1: Feature Extraction (freeze backbone)
    # ============================================================
    print("\n" + "-"*60)
    print("STAGE 1: Feature Extraction (backbone frozen)")
    print("-"*60)
    
    # Freeze all layers except classifier
    for name, param in densenet_model.named_parameters():
        if 'classifier' not in name:  # DenseNet uses 'classifier' for final layer
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in densenet_model.parameters() if p.requires_grad)
    print(f"Trainable parameters (stage 1): {trainable:,}")
    
    # Training components
    criterion = nn.BCEWithLogitsLoss()  # Includes sigmoid
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, densenet_model.parameters()),
        lr=CONFIG['learning_rate_stage1']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=CONFIG['reduce_lr_factor'], 
        patience=CONFIG['reduce_lr_patience']
    )
    
    # Train stage 1
    history_s1 = train_model(
        densenet_model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        num_epochs=CONFIG['epochs_stage1'],
        device=device,
        save_path=MODELS_DIR / 'densenet121_pytorch_s1.pt',
        model_name='densenet121',
        stage='stage1',
        patience=5,
        use_mlflow=USE_MLFLOW
    )
    
    # ============================================================
    # STAGE 2: Fine-Tuning (unfreeze some layers)
    # ============================================================
    print("\n" + "-"*60)
    print("STAGE 2: Fine-Tuning (unfreeze last layers)")
    print("-"*60)
    
    # Unfreeze all layers
    for param in densenet_model.parameters():
        param.requires_grad = True
    
    trainable = sum(p.numel() for p in densenet_model.parameters() if p.requires_grad)
    print(f"Trainable parameters (stage 2): {trainable:,}")
    
    # Lower learning rate for fine-tuning
    optimizer = optim.Adam(
        densenet_model.parameters(),
        lr=CONFIG['learning_rate_stage2']  # 10x lower
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=CONFIG['reduce_lr_factor'], 
        patience=CONFIG['reduce_lr_patience']
    )
    
    # Train stage 2
    history_s2 = train_model(
        densenet_model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        num_epochs=CONFIG['epochs_stage2'],
        device=device,
        save_path=densenet_path,
        model_name='densenet121',
        stage='stage2',
        patience=CONFIG['early_stopping_patience'],
        use_mlflow=USE_MLFLOW
    )
    
    # Log model to MLflow
    if USE_MLFLOW:
        mlflow.pytorch.log_model(densenet_model, "model")
        mlflow.log_artifact(str(densenet_path))
        mlflow.end_run()
    
    print("\n✓ DenseNet121 training complete")
else:
    if 'densenet121' not in MODELS_TO_TRAIN:
        print("\nSkipping DenseNet121 (not in MODELS_TO_TRAIN)")
    else:
        print("\nSkipping DenseNet121 training (loading saved model later)")

## 8. Model 2: ConvNeXt-Tiny

**ConvNeXt** (2022) - Modern pure ConvNet that rivals Vision Transformers

**Key Innovations:**
- Modernized ResNet design using Transformer principles
- Depthwise convolutions, LayerNorm, GELU activation
- Competitive with Swin Transformers at lower cost
- Published in "A ConvNet for the 2020s" (Facebook AI Research)

**Why ConvNeXt?**
- State-of-art pure CNN (no attention mechanisms)
- More efficient than ViTs for medical imaging
- Better inductive bias for spatial data
- Unavailable in TensorFlow keras.applications

---

In [ ]:
if not SKIP_TRAINING and 'convnext_tiny' in MODELS_TO_TRAIN:
    print("="*60)
    print("MODEL 2: ConvNeXt-Tiny (PyTorch)")
    print("="*60)
    
    # Start MLflow run for ConvNeXt
    if USE_MLFLOW:
        mlflow_run = mlflow.start_run(run_name=f"{RUN_NAME}_convnext")
        mlflow.log_params({
            'model': 'convnext_tiny',
            'framework': 'pytorch',
            'batch_size': CONFIG['batch_size'],
            'epochs_stage1': CONFIG['epochs_stage1'],
            'epochs_stage2': CONFIG['epochs_stage2'],
            'lr_stage1': CONFIG['learning_rate_stage1'],
            'lr_stage2': CONFIG['learning_rate_stage2'],
            'dropout_rate': CONFIG['dropout_rate'],
            'use_sample': CONFIG['use_sample'],
        })
    
    # Create model
    convnext_model = create_transfer_model('convnext_tiny', num_classes=CONFIG['num_classes'])
    convnext_model = convnext_model.to(device)
    
    print(f"\nModel: ConvNeXt-Tiny")
    print(f"Parameters: {sum(p.numel() for p in convnext_model.parameters()):,}")
    
    # STAGE 1: Feature Extraction
    print("\n" + "-"*60)
    print("STAGE 1: Feature Extraction")
    print("-"*60)
    
    # Freeze backbone (ConvNeXt uses 'head' for classifier)
    for name, param in convnext_model.named_parameters():
        if 'head' not in name:
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in convnext_model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, convnext_model.parameters()),
        lr=CONFIG['learning_rate_stage1']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=CONFIG['reduce_lr_factor'], 
        patience=CONFIG['reduce_lr_patience']
    )
    
    history_s1 = train_model(
        convnext_model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        num_epochs=CONFIG['epochs_stage1'],
        device=device,
        save_path=MODELS_DIR / 'convnext_tiny_pytorch_s1.pt',
        model_name='convnext_tiny',
        stage='stage1',
        patience=5,
        use_mlflow=USE_MLFLOW
    )
    
    # STAGE 2: Fine-Tuning
    print("\n" + "-"*60)
    print("STAGE 2: Fine-Tuning")
    print("-"*60)
    
    for param in convnext_model.parameters():
        param.requires_grad = True
    
    trainable = sum(p.numel() for p in convnext_model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
    
    optimizer = optim.Adam(convnext_model.parameters(), lr=CONFIG['learning_rate_stage2'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=CONFIG['reduce_lr_factor'], 
        patience=CONFIG['reduce_lr_patience']
    )
    
    history_s2 = train_model(
        convnext_model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        num_epochs=CONFIG['epochs_stage2'],
        device=device,
        save_path=MODELS_DIR / 'convnext_tiny_pytorch_best.pt',
        model_name='convnext_tiny',
        stage='stage2',
        patience=CONFIG['early_stopping_patience'],
        use_mlflow=USE_MLFLOW
    )
    
    # Log model to MLflow
    if USE_MLFLOW:
        mlflow.pytorch.log_model(convnext_model, "model")
        mlflow.log_artifact(str(MODELS_DIR / 'convnext_tiny_pytorch_best.pt'))
        mlflow.end_run()
    
    print("\n✓ ConvNeXt-Tiny training complete")
else:
    if 'convnext_tiny' not in MODELS_TO_TRAIN:
        print("\nSkipping ConvNeXt-Tiny (not in MODELS_TO_TRAIN)")
    else:
        print("\nSkipping ConvNeXt-Tiny training (loading saved model later)")

## 9. Model 3: EfficientNetV2-S

**EfficientNetV2** (2021) - Improved EfficientNet by Google Brain

**Improvements over EfficientNet:**
- Fused-MBConv blocks (faster training)
- Progressive training (adaptive image sizes)
- Better parameter efficiency
- Up to 11x faster training than original EfficientNet

**Why EfficientNetV2?**
- Best accuracy/parameter trade-off
- Production-ready (efficient inference)
- Better than EfficientNetB3 used in TF notebook 07
- Google's latest ConvNet before switching to ViTs

---

In [ ]:
if not SKIP_TRAINING and 'efficientnetv2_s' in MODELS_TO_TRAIN:
    print("="*60)
    print("MODEL 3: EfficientNetV2-S (PyTorch)")
    print("="*60)
    
    # Start MLflow run for EfficientNetV2
    if USE_MLFLOW:
        mlflow_run = mlflow.start_run(run_name=f"{RUN_NAME}_efficientnetv2")
        mlflow.log_params({
            'model': 'efficientnetv2_s',
            'framework': 'pytorch',
            'batch_size': CONFIG['batch_size'],
            'epochs_stage1': CONFIG['epochs_stage1'],
            'epochs_stage2': CONFIG['epochs_stage2'],
            'lr_stage1': CONFIG['learning_rate_stage1'],
            'lr_stage2': CONFIG['learning_rate_stage2'],
            'dropout_rate': CONFIG['dropout_rate'],
            'use_sample': CONFIG['use_sample'],
        })
    
    # Create model
    efficientnet_model = create_transfer_model('tf_efficientnetv2_s', num_classes=CONFIG['num_classes'])
    efficientnet_model = efficientnet_model.to(device)
    
    print(f"\nModel: EfficientNetV2-S")
    print(f"Parameters: {sum(p.numel() for p in efficientnet_model.parameters()):,}")
    
    # STAGE 1: Feature Extraction
    print("\n" + "-"*60)
    print("STAGE 1: Feature Extraction")
    print("-"*60)
    
    # Freeze backbone (EfficientNet uses 'classifier' for final layer)
    for name, param in efficientnet_model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False
    
    trainable = sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, efficientnet_model.parameters()),
        lr=CONFIG['learning_rate_stage1']
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=CONFIG['reduce_lr_factor'], 
        patience=CONFIG['reduce_lr_patience']
    )
    
    history_s1 = train_model(
        efficientnet_model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        num_epochs=CONFIG['epochs_stage1'],
        device=device,
        save_path=MODELS_DIR / 'efficientnetv2_s_pytorch_s1.pt',
        model_name='efficientnetv2_s',
        stage='stage1',
        patience=5,
        use_mlflow=USE_MLFLOW
    )
    
    # STAGE 2: Fine-Tuning
    print("\n" + "-"*60)
    print("STAGE 2: Fine-Tuning")
    print("-"*60)
    
    for param in efficientnet_model.parameters():
        param.requires_grad = True
    
    trainable = sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")
    
    optimizer = optim.Adam(efficientnet_model.parameters(), lr=CONFIG['learning_rate_stage2'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=CONFIG['reduce_lr_factor'], 
        patience=CONFIG['reduce_lr_patience']
    )
    
    history_s2 = train_model(
        efficientnet_model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        num_epochs=CONFIG['epochs_stage2'],
        device=device,
        save_path=MODELS_DIR / 'efficientnetv2_s_pytorch_best.pt',
        model_name='efficientnetv2_s',
        stage='stage2',
        patience=CONFIG['early_stopping_patience'],
        use_mlflow=USE_MLFLOW
    )
    
    # Log model to MLflow
    if USE_MLFLOW:
        mlflow.pytorch.log_model(efficientnet_model, "model")
        mlflow.log_artifact(str(MODELS_DIR / 'efficientnetv2_s_pytorch_best.pt'))
        mlflow.end_run()
    
    print("\n✓ EfficientNetV2-S training complete")
else:
    if 'efficientnetv2_s' not in MODELS_TO_TRAIN:
        print("\nSkipping EfficientNetV2-S (not in MODELS_TO_TRAIN)")
    else:
        print("\nSkipping EfficientNetV2-S training (loading saved model later)")

## 10. Evaluate PyTorch Models on Test Set

In [ ]:
# Load best models
print("Loading best PyTorch models...")

densenet_best = create_transfer_model('densenet121', num_classes=CONFIG['num_classes'])
densenet_best.load_state_dict(torch.load(MODELS_DIR / 'densenet121_pytorch_best.pt'))
densenet_best = densenet_best.to(device)

convnext_best = create_transfer_model('convnext_tiny', num_classes=CONFIG['num_classes'])
convnext_best.load_state_dict(torch.load(MODELS_DIR / 'convnext_tiny_pytorch_best.pt'))
convnext_best = convnext_best.to(device)

efficientnet_best = create_transfer_model('tf_efficientnetv2_s', num_classes=CONFIG['num_classes'])
efficientnet_best.load_state_dict(torch.load(MODELS_DIR / 'efficientnetv2_s_pytorch_best.pt'))
efficientnet_best = efficientnet_best.to(device)

print("✓ All PyTorch models loaded")

In [ ]:
def evaluate_model(model, loader, device):
    """
    Evaluate model on test set.
    
    Returns:
        Dictionary with loss, accuracy, and AUC
    """
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    test_loss = running_loss / len(loader.dataset)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    test_acc = accuracy_score(all_labels.ravel(), (all_preds > 0.5).ravel())
    test_auc = roc_auc_score(all_labels, all_preds, average='macro')
    
    return {
        'loss': float(test_loss),
        'accuracy': float(test_acc),
        'auc': float(test_auc)
    }

print("✓ evaluate_model function defined")

In [ ]:
# Evaluate all PyTorch models
print("="*60)
print("TEST SET EVALUATION (PyTorch Models)")
print("="*60)

pytorch_results = {}

for model_name, model in [
    ('DenseNet121', densenet_best),
    ('ConvNeXt-Tiny', convnext_best),
    ('EfficientNetV2-S', efficientnet_best)
]:
    print(f"\n{model_name}:")
    results = evaluate_model(model, test_loader, device)
    pytorch_results[model_name] = results
    
    print(f"  Loss: {results['loss']:.4f}")
    print(f"  Accuracy: {results['accuracy']:.4f}")
    print(f"  AUC: {results['auc']:.4f}")

print("\n" + "="*60)

## 11. Compare PyTorch vs TensorFlow vs Baseline Models

In [ ]:
# Load results from previous notebooks
with open(OUTPUTS_DIR / 'reports' / '05_baseline_models_results.json', 'r') as f:
    baseline_results = json.load(f)

with open(OUTPUTS_DIR / 'reports' / '06_cnn_results.json', 'r') as f:
    cnn_results = json.load(f)

with open(OUTPUTS_DIR / 'reports' / '07_transfer_learning_results.json', 'r') as f:
    tf_transfer_results = json.load(f)

print("✓ Loaded results from previous notebooks")

In [ ]:
# Create comprehensive comparison table
comparison_data = {
    'Framework': [],
    'Approach': [],
    'Model': [],
    'Test AUC': [],
    'Parameters': []
}

# Baseline models
comparison_data['Framework'].append('scikit-learn')
comparison_data['Approach'].append('Baseline (Hand-crafted)')
comparison_data['Model'].append(baseline_results['best_model']['name'])
comparison_data['Test AUC'].append(baseline_results['best_model']['test_avg_auc'])
comparison_data['Parameters'].append('~1.8K features')

# Custom CNN (TensorFlow)
comparison_data['Framework'].append('TensorFlow')
comparison_data['Approach'].append('Custom CNN')
comparison_data['Model'].append('4-block CNN')
comparison_data['Test AUC'].append(cnn_results['test_performance']['overall']['auc'])
comparison_data['Parameters'].append(f"{cnn_results['architecture']['total_params']:,}")

# TensorFlow transfer learning
for model_name in ['resnet50', 'densenet121', 'efficientnetb3']:
    comparison_data['Framework'].append('TensorFlow')
    comparison_data['Approach'].append('Transfer Learning')
    comparison_data['Model'].append(model_name.upper().replace('RESNET', 'ResNet').replace('DENSENET', 'DenseNet').replace('EFFICIENTNETB', 'EfficientNetB'))
    comparison_data['Test AUC'].append(tf_transfer_results['models'][model_name]['auc'])
    
    if model_name == 'resnet50':
        comparison_data['Parameters'].append('~25M')
    elif model_name == 'densenet121':
        comparison_data['Parameters'].append('~8M')
    else:
        comparison_data['Parameters'].append('~12M')

# PyTorch transfer learning
for model_name in ['DenseNet121', 'ConvNeXt-Tiny', 'EfficientNetV2-S']:
    comparison_data['Framework'].append('PyTorch')
    comparison_data['Approach'].append('Transfer Learning')
    comparison_data['Model'].append(model_name)
    comparison_data['Test AUC'].append(pytorch_results[model_name]['auc'])
    
    if model_name == 'DenseNet121':
        comparison_data['Parameters'].append('~8M')
    elif model_name == 'ConvNeXt-Tiny':
        comparison_data['Parameters'].append('~29M')
    else:
        comparison_data['Parameters'].append('~22M')

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*60)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*60)
print(comparison_df.to_string(index=False))
print("\n" + "="*60)

# Find best overall model
best_idx = comparison_df['Test AUC'].idxmax()
best_model_info = comparison_df.iloc[best_idx]

print(f"\n🏆 BEST MODEL OVERALL: {best_model_info['Model']}")
print(f"   Framework: {best_model_info['Framework']}")
print(f"   Approach: {best_model_info['Approach']}")
print(f"   Test AUC: {best_model_info['Test AUC']:.4f}")
print(f"   Parameters: {best_model_info['Parameters']}")

## 12. Visualization

In [ ]:
# Plot comprehensive comparison
fig, ax = plt.subplots(figsize=(14, 8))

# Color by framework
colors = []
for fw in comparison_df['Framework']:
    if fw == 'scikit-learn':
        colors.append('#e74c3c')  # Red
    elif fw == 'TensorFlow':
        colors.append('#3498db')  # Blue
    else:  # PyTorch
        colors.append('#2ecc71')  # Green

bars = ax.barh(range(len(comparison_df)), comparison_df['Test AUC'], color=colors, alpha=0.8)

# Add value labels
for i, (bar, auc) in enumerate(zip(bars, comparison_df['Test AUC'])):
    ax.text(auc + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{auc:.3f}', va='center', fontweight='bold', fontsize=9)

# Customize plot
ax.set_yticks(range(len(comparison_df)))
ax.set_yticklabels(comparison_df['Model'])
ax.set_xlabel('Test AUC', fontweight='bold', fontsize=12)
ax.set_title('Complete Model Comparison: Baseline → TensorFlow → PyTorch', 
             fontweight='bold', fontsize=14)
ax.set_xlim((0, 1))
ax.grid(axis='x', alpha=0.3)

# Add framework labels
for i, fw in enumerate(comparison_df['Framework']):
    ax.text(-0.02, i, fw, ha='right', va='center', 
            fontsize=8, style='italic', color='gray')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='scikit-learn'),
    Patch(facecolor='#3498db', label='TensorFlow'),
    Patch(facecolor='#2ecc71', label='PyTorch')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '07b_pytorch_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved comparison visualization")

## 13. Save PyTorch Results

In [ ]:
# Compile all results
pytorch_transfer_results = {
    'config': CONFIG,
    'framework': 'PyTorch',
    'timm_version': timm.__version__,
    'torch_version': torch.__version__,
    'device': str(device),
    'models': {
        'densenet121': pytorch_results['DenseNet121'],
        'convnext_tiny': pytorch_results['ConvNeXt-Tiny'],
        'efficientnetv2_s': pytorch_results['EfficientNetV2-S']
    },
    'best_pytorch_model': {
        'name': max(pytorch_results.keys(), key=lambda k: pytorch_results[k]['auc']),
        'auc': max(pytorch_results.values(), key=lambda v: v['auc'])['auc']
    },
    'best_overall': {
        'name': best_model_info['Model'],
        'framework': best_model_info['Framework'],
        'auc': float(best_model_info['Test AUC']),
        'parameters': best_model_info['Parameters']
    }
}

# Save to JSON
with open(OUTPUTS_DIR / 'reports' / '07b_pytorch_transfer_learning_results.json', 'w') as f:
    json.dump(pytorch_transfer_results, f, indent=2)

print(f"✓ Saved PyTorch results to {OUTPUTS_DIR / 'reports' / '07b_pytorch_transfer_learning_results.json'}")

## 14. Summary

In [ ]:
print("="*60)
print("  ✅ Notebook 07b Complete: Transfer Learning with PyTorch")
print("="*60)

print("\n🏗️ Models Trained (PyTorch):")
print("  1. DenseNet121 (~8M params)")
print("  2. ConvNeXt-Tiny (~29M params)")
print("  3. EfficientNetV2-S (~22M params)")

print("\n📊 Training Strategy:")
print(f"  Stage 1: Feature extraction ({CONFIG['epochs_stage1']} epochs, backbone frozen)")
print(f"  Stage 2: Fine-tuning ({CONFIG['epochs_stage2']} epochs, full model trainable)")

print("\n🎯 PyTorch Model Performance:")
for model_name, results in pytorch_results.items():
    print(f"  {model_name}: AUC = {results['auc']:.4f}")

print(f"\n🏆 Best Overall Model: {best_model_info['Model']}")
print(f"   Framework: {best_model_info['Framework']}")
print(f"   Test AUC: {best_model_info['Test AUC']:.4f}")

print("\n📈 Performance Evolution:")
baseline_auc = baseline_results['best_model']['test_avg_auc']
tf_best_auc = max([tf_transfer_results['models'][m]['auc'] for m in tf_transfer_results['models'].keys()])
pytorch_best_auc = pytorch_transfer_results['best_pytorch_model']['auc']

print(f"  Baseline (scikit-learn): {baseline_auc:.4f}")
print(f"  TensorFlow Transfer: {tf_best_auc:.4f} ({((tf_best_auc - baseline_auc) / baseline_auc * 100):+.1f}%)")
print(f"  PyTorch Transfer: {pytorch_best_auc:.4f} ({((pytorch_best_auc - baseline_auc) / baseline_auc * 100):+.1f}%)")

print("\n📁 Generated Files:")
print(f"  {MODELS_DIR / 'densenet121_pytorch_best.pt'}")
print(f"  {MODELS_DIR / 'convnext_tiny_pytorch_best.pt'}")
print(f"  {MODELS_DIR / 'efficientnetv2_s_pytorch_best.pt'}")
print(f"  {OUTPUTS_DIR / 'reports' / '07b_pytorch_transfer_learning_results.json'}")
print(f"  {FIGURES_DIR / '07b_pytorch_comparison.png'}")

print("\n💡 Key Insights:")
print("  - PyTorch provides access to 500+ models via timm library")
print("  - ConvNeXt-Tiny: Modern pure ConvNet architecture (2022)")
print("  - EfficientNetV2: Improved efficiency vs EfficientNetB3")
print("  - Manual training loops offer explicit control and debugging")
print("  - Framework choice depends on task: TF (production), PyTorch (research)")

print("\n🔍 Framework Comparison (TensorFlow vs PyTorch):")
print(f"  TensorFlow best: {tf_best_auc:.4f}")
print(f"  PyTorch best: {pytorch_best_auc:.4f}")
print(f"  Difference: {abs(tf_best_auc - pytorch_best_auc):.4f} ({abs((pytorch_best_auc - tf_best_auc) / tf_best_auc * 100):.1f}%)")

print("\n⏭️  Next: Notebook 08 - Model Evaluation & Interpretation!")
print("  Grad-CAM visualizations, per-disease analysis, error analysis")